# Simple Neural Network With PyTorch

In this lab, we'll build two neural networks with PyTorch, writing their parameters, their loss and their training loop ourselves: a network without a hidden layer first, then a multilayer perceptron. The next lab builds the same networks with PyTorch Lightning, which writes the loop for us.

The data is MNIST, the *Hello World!* dataset of machine learning: 28 $\times$ 28 pixel grayscale images of the digits 0 to 9. Each image comes with a label, the digit the image is supposed to represent.

The steps:

1. Get started with PyTorch tensors
2. Explore and prepare the data, set a validation set aside
3. Write a network without a hidden layer, its loss, its accuracy and its `DataLoader`
4. Train it by gradient descent written by hand, then with a PyTorch optimizer
5. Compile the model with `torch.compile`, and measure what compiling costs and brings
6. Move on to a multilayer perceptron and regularize it
7. Follow its training runs in TensorBoard, choose its learning rate on the validation set
8. Evaluate it on the test set, save it and load it back

This notebook needs no GPU: each training run takes from a few seconds to a few minutes on Colab's CPU.

## Imports

In [ ]:
import time
from collections.abc import Callable

import matplotlib.pyplot as plt
import numpy as np
import seaborn
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter

# Computations run on the GPU if there is one
device = "cuda" if torch.cuda.is_available() else "cpu"

PyTorch provides the tensors, the gradients, the optimizers and the loading of data in batches; torchvision, the MNIST dataset; `torch.utils.tensorboard`, the training logs TensorBoard displays. [`matplotlib`](https://matplotlib.org/), [`numpy`](https://numpy.org/) and [`seaborn`](https://seaborn.pydata.org/) are used for plots and simple array operations.

## Getting Started With Tensors

On Colab, PyTorch is already installed. On your machine, install it with `pip install torch torchvision`. On Linux without an NVIDIA GPU, add `--index-url https://download.pytorch.org/whl/cpu`: without it, pip downloads the CUDA version and several GB of useless NVIDIA libraries (slide "Installing PyTorch"). For another system or another GPU, [pytorch.org](https://pytorch.org/get-started/locally/) gives the suited command.

Before building networks, let's handle PyTorch tensors, whose API is very close to NumPy's (slides "NumPy and PyTorch" and "Computing on GPU").

1. Create a tensor `a` of shape (2, 3) holding the integers from 0 to 5 (`torch.arange`, then `reshape`). What is its `dtype`? Convert it to floats (`.float()`), then create a tensor `b` of shape (3,) filled with 1s (`torch.ones`).
2. Compute `a + b` and the matrix product `a @ b`. What are the shapes of the results? Why does `a + torch.ones(2)` fail? An operation that fails raises a `RuntimeError`: catch it with `try`/`except` to display its message.
3. Extract the second column of `a`, then its last row. Add a dimension of size 1 in front of `a` with `unsqueeze`: what shape do you get? What is such a dimension for with a PyTorch model?
4. Convert `a` to a NumPy array (`.numpy()`), then back to a tensor (`torch.from_numpy`). Copy `a` to `device` (defined in the imports cell) and display its `device` attribute.

**Useful documentation**

- [`torch.arange`](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)
- [`torch.Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)
- [`torch.Tensor.unsqueeze`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.unsqueeze.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `torch.arange(6).reshape(2, 3)`, then `a.float()`
- `a[:, 1]`: the second column; `a[-1]`: the last row
- `a.unsqueeze(0)`: a dimension of size 1 at position 0
- `a.to(device)`: a copy on `device`; its `device` attribute tells where it is
- An operation that fails, in a `try`:

```python
try:
  a + torch.ones(2)
except RuntimeError as error:
  print(error)
```

A skeleton to copy into the exercise cell, replacing each `...`:

```python
a = torch.arange(...).reshape(...)
print(a.dtype)
a = a.float()
b = ...

print((a + b).shape, (a @ b).shape)
try:
  ...  # a + torch.ones(2)
except RuntimeError as error:
  print(error)

print(a[:, 1], ...)  # second column, last row
print(a.unsqueeze(...).shape)

print(type(a.numpy()), torch.from_numpy(...).dtype)
print(a.to(...).device)
```

### Solution

In [ ]:
a = torch.arange(6).reshape(2, 3)
print(a, a.dtype)
a = a.float()
b = torch.ones(3)

# b is broadcast over each row of a
print(a + b, (a + b).shape)
# Matrix-vector product: the sum of each row of a
print(a @ b, (a @ b).shape)
try:
  a + torch.ones(2)
except RuntimeError as error:
  print("a + torch.ones(2):", error)

print(a[:, 1])  # second column
print(a[-1])  # last row
print(a.unsqueeze(0).shape)  # batch dimension of size 1

a_numpy = a.numpy()
print(type(a_numpy), torch.from_numpy(a_numpy).dtype)
print(a.to(device).device)

- `torch.arange` creates 64-bit integers (`torch.int64`); neural networks compute with floats, `torch.float32` by default.
- `a + b` has shape (2, 3) and `a @ b` shape (2,). For broadcasting, dimensions are compared starting from the last one: they must be equal, or one of them must be 1 or missing. (2, 3) and (3,) are compatible, but not (2, 3) and (2,): it would take the shape (2, 1), `torch.ones(2).unsqueeze(1)`.
- `a.unsqueeze(0)` has shape (1, 2, 3). PyTorch models process batches, the first dimension being the examples': to predict on a single example, we add a batch dimension of size 1 to it.
- `a.numpy()` shares the tensor's memory, without a copy: this requires a tensor on CPU (`.cpu()`) and outside the autograd graph (`.detach()`). `torch.from_numpy` keeps the NumPy type (`float32` here, but `float64` for a NumPy array created by default).
- Without a GPU, `a.to(device).device` is `cpu`; with a GPU, `cuda:0`.

## Fetching the Data

In [ ]:
train_data = torchvision.datasets.MNIST("data", train=True, download=True)
test_data = torchvision.datasets.MNIST("data", train=False, download=True)
X_train, y_train = train_data.data, train_data.targets
X_test, y_test = test_data.data, test_data.targets

## Let's Look at the Data

Use the following cell to explore the tensors `X_train`, `y_train`, `X_test` and `y_test`: their shape (`shape`), their type (`dtype`), their extreme values, and a few images with `plt.imshow`.

Try to answer the following questions:

- How many images are there in total in this dataset?
- In what form are the images stored? The labels?
- Are the classes balanced?

**Useful documentation**

- [`torch.bincount`](https://docs.pytorch.org/docs/stable/generated/torch.bincount.html)
- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `X_train.shape`, `X_train.dtype`, `X_train.min()`, `X_train.max()`: shape, type and extreme values
- `torch.bincount(y_train)`: the number of examples of each class, from 0 to 9
- `plt.imshow(X_train[0], cmap="gray_r")`: the first image, black on white

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print(X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print(y_train.shape, ...)
print(X_test.shape, ...)

print(torch.bincount(...))

plt.imshow(..., cmap="gray_r")
plt.title(f"Label: {int(y_train[0])}")
plt.show()
```

### Solution

In [ ]:
print(f"Training images: {tuple(X_train.shape)}, {X_train.dtype}")
print(f"Pixel values: from {int(X_train.min())} to {int(X_train.max())}")
print(f"Training labels: {tuple(y_train.shape)}, {y_train.dtype}")
print(f"Test images and labels: {tuple(X_test.shape)}, {tuple(y_test.shape)}")
print(f"Examples per class: {torch.bincount(y_train).tolist()}")

plt.imshow(X_train[0], cmap="gray_r")
plt.title(f"First example of the dataset ({int(y_train[0])})")
plt.show()

seaborn.countplot(x=y_train.numpy())
plt.title("Counts of the different classes (digits)")
plt.ylabel("Count")
plt.xlabel("Digit")
plt.show()

- 70,000 images: 60,000 for training and 10,000 for testing.
- The images are a `torch.uint8` tensor of shape (60,000, 28, 28): an integer from 0 (background) to 255 (ink) per pixel. The labels are a `torch.int64` tensor of shape (60,000,), one digit from 0 to 9 per image.
- The classes are roughly balanced: from 5,421 examples (the 5) to 6,742 (the 1). Accuracy, the proportion of right answers, is therefore a relevant metric.

## Mean Image of Each Digit

*Compute the mean image of each digit: for each label `k` from 0 to 9, the pixel-wise mean of the training images with label `k`. Display the 10 mean images side by side, each with its digit as title. What do they show?*

**Useful documentation**

- [`torch.mean`](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)
- [`matplotlib.pyplot.subplots`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html)

In [ ]:
# A row of 10 subplots, one per digit
f, ax = plt.subplots(1, 10, figsize=(15, 2))

# Your code here

### Additional hints

The calls and parameters to use:

- `y_train == k`: a mask, a tensor of booleans that is `True` for the examples with label `k`
- `X_train[mask]`: only the images for which the mask is `True`, a tensor of shape (number of images with label `k`, 28, 28)
- `.float()`: the pixels are `torch.uint8` integers, whose mean PyTorch refuses to compute
- `.mean(dim=0)`: the mean over dimension 0, the examples'; what remains is an image of shape (28, 28)
- `ax[k].imshow(image, cmap="gray_r")`, `ax[k].set_title(...)` and `ax[k].axis("off")`: the image, its title and no axes

A skeleton to copy into the exercise cell, replacing each `...`:

```python
f, ax = plt.subplots(1, 10, figsize=(15, 2))
for k in range(10):
  mean_image = X_train[...].float().mean(dim=...)
  ax[k].imshow(..., cmap="gray_r")
  ax[k].set_title(...)
  ax[k].axis("off")
```

### Solution

In [ ]:
f, ax = plt.subplots(1, 10, figsize=(15, 2))

for k in range(10):
  # The mask y_train == k keeps the images with label k; their mean over
  # dimension 0, the examples', is an image of 28 x 28 pixels
  mean_image = X_train[y_train == k].float().mean(dim=0)
  ax[k].imshow(mean_image, cmap="gray_r")
  ax[k].set_title(str(k))
  ax[k].axis("off")

Each mean image shows the typical shape of its digit: dark where nearly all the handwritings go, blurred where they vary from one person to the next. The 1, a more or less slanted stroke, gives the narrowest image.

The network without a hidden layer that comes next computes something close: it compares each image with one template per digit, a column of its weight matrix, and picks the digit whose template it resembles most.

## Transforming the Data

Three transformations prepare the data for the network:

1. Flatten the images: a network without a hidden layer takes one vector per example, of shape `(_, 28²)` rather than `(_, 28, 28)`, where `_` is the number of examples.
2. Bring the pixels into the $[0, 1]$ range rather than between $0$ and $255$.
3. Copy the images and labels to `device`, where the model will compute (slide "Computing on GPU").

*Apply these three transformations to `X_train` and `X_test`, and the third one to `y_train` and `y_test`.*

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `X.reshape(-1, 28 * 28)`: `-1` lets PyTorch compute the number of rows
- `X / 255`: values between 0 and 1; dividing an integer tensor gives `torch.float32` values
- `X.to(device)`: the copy on `device`

Expected: `X_train.shape` is `torch.Size([60000, 784])`, `X_train.dtype` is `torch.float32` and `X_train.max()` is 1.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
X_train = (X_train.reshape(-1, ...) / ...).to(device)
X_test = ...
y_train, y_test = y_train.to(...), ...
print(X_train.shape, X_train.dtype, X_train.max())
```

### Solution

In [ ]:
input_dim = 28 * 28

# Flattening: a vector of input_dim pixels per image, then scaling into [0, 1]
# and copying to the compute device
X_train = (X_train.reshape(-1, input_dim) / 255).to(device)
X_test = (X_test.reshape(-1, input_dim) / 255).to(device)
y_train, y_test = y_train.to(device), y_test.to(device)
print(X_train.shape, X_train.dtype, X_train.max())

## Train / Validation Split

To choose the hyperparameters (learning rate, network size…), we compare models on a **validation** set. The test set is kept for the final evaluation: using it to make choices would make that evaluation too optimistic.

We set the last 10,000 training examples aside for validation:

In [ ]:
X_train, X_val = X_train[:50_000], X_train[50_000:]
y_train, y_val = y_train[:50_000], y_train[50_000:]
print(X_train.shape, X_val.shape, X_test.shape)

## Neural Network Without a Hidden Layer

A neural network without a hidden layer computes $\text{softmax}(XW + b)$: this is **multinomial logistic regression**. `W` is the weight matrix, multiplied with the input, and `b` the bias vector, added next. $XW + b$ gives 10 scores per example, the **logits**, which the softmax turns into probabilities.

In PyTorch, the model stops at the logits, without a final softmax: the loss function computes the logarithm of the softmax itself (slide "Classic Pitfall: Softmax and Cross-Entropy").

### Initializing the Parameters

*Complete the `create_simple_nn_weights(n_inputs, n_outputs)` function, which returns two [PyTorch parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) created on `device`: `W`, of shape `(n_inputs, n_outputs)`, drawn from a normal distribution (`torch.randn`) and divided by $\sqrt{n_\text{inputs}}$; `b`, a vector of zeros of shape `(n_outputs,)`.*

This division gives the weights a variance of $1 / n_\text{inputs}$: the layer's output, a sum of $n_\text{inputs}$ terms, stays of the same order of magnitude as its inputs, whatever their number (slide "Weight Initialization").

**Useful documentation**

- [`torch.randn`](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)
- [`torch.zeros`](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

In [ ]:
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  pass  # Your code here

#### Additional hints

- `torch.randn(n_inputs, n_outputs, device=device)`: a tensor with mean 0 and standard deviation 1, created directly on `device`
- `n_inputs**0.5`: the square root of `n_inputs`
- `torch.zeros(n_outputs, device=device)`: the biases, zero at first
- `nn.Parameter(tensor)`: a tensor whose gradient PyTorch computes, and that an `nn.Module` recognizes as a parameter

Expected: `W` has shape (784, 10) and a standard deviation (`W.detach().std()`) close to $1 / 28 \approx 0.036$.

```python
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  W = torch.randn(..., ..., device=device) / ...
  b = torch.zeros(..., device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = create_simple_nn_weights(784, 10)
print(W.shape, b.shape, W.detach().std())
```

#### Solution

In [ ]:
def create_simple_nn_weights(
  n_inputs: int, n_outputs: int
) -> tuple[nn.Parameter, nn.Parameter]:
  # Variance 1 / n_inputs: outputs of the order of 1, whatever n_inputs
  W = torch.randn(n_inputs, n_outputs, device=device) / n_inputs**0.5
  b = torch.zeros(n_outputs, device=device)
  return nn.Parameter(W), nn.Parameter(b)


W, b = create_simple_nn_weights(784, 10)
print(W.shape, b.shape, W.detach().std())

### Defining the Model

*Complete the `forward` method of `SimpleNN`, which returns the logits $XW + b$ of each example.*

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Your code here

#### Additional hints

- `X @ self.W + self.b`: the matrix product, then adding the biases, broadcast over each row

Expected: `SimpleNN()(X_train[:5])` has shape (5, 10), and the softmax of each row (`torch.softmax(..., dim=-1)`) sums to 1.

```python
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    return ... @ ... + ...
```

#### Solution

In [ ]:
class SimpleNN(nn.Module):
  def __init__(self, n_inputs: int = 28**2, n_outputs: int = 10) -> None:
    super().__init__()
    self.W, self.b = create_simple_nn_weights(n_inputs, n_outputs)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # The logits: one score per class, without softmax
    return X @ self.W + self.b


logits = SimpleNN()(X_train[:5])
# The softmax turns them into probabilities, summing to 1 on each row
print(logits.shape, torch.softmax(logits, dim=-1).sum(dim=-1))

### Computing the Loss Function

The standard loss function for multi-class classification is cross-entropy: the mean, over the examples, of the opposite of the log-probability the model gives the right class. The log-probabilities are computed from the logits with `torch.log_softmax`: the logarithm of the softmax, computed in one go, is more numerically stable than the softmax followed by a logarithm.

*Complete the `categorical_crossentropy` function, which takes the model's logits and the integer labels `y` (from 0 to 9), in the order of `nn.functional.cross_entropy`, by writing the computation yourself: the log-probabilities, the one of each example's right class, then the opposite of their mean. Compare your result with that of `nn.functional.cross_entropy` (slide "Classic Pitfall: Softmax and Cross-Entropy"). What value does the loss take for a model that hasn't learned yet?*

**Useful documentation**

- [`torch.log_softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)

In [ ]:
def categorical_crossentropy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
  pass  # Your code here

#### Additional hints

- `torch.log_softmax(logits, dim=-1)`: the log-probabilities, one row per example, one column per class
- `log_probs[torch.arange(len(y), device=y.device), y]`: indexed by two integer tensors, `log_probs` gives, for each row `i`, its column `y[i]`: the log-probability of each example's right class (`device=y.device` creates the row numbers where the labels are)
- `.mean()`, preceded by a minus sign: the opposite of the mean
- `nn.functional.cross_entropy(logits, y)` does this computation in one go, `log_softmax` included: it's the one used in practice. Its arguments come in the same order, the logits first.

A model that hasn't learned gives roughly the same probability, 1/10, to each class: its loss is about $-\log(1/10) = \log 10 \approx 2.3$, and both functions give the same value.

```python
def categorical_crossentropy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
  log_probs = torch.log_softmax(..., dim=...)
  return -log_probs[torch.arange(len(y), device=y.device), ...].mean()


logits = SimpleNN()(X_train)
print(categorical_crossentropy(logits, y_train))
print(nn.functional.cross_entropy(..., ...))
```

#### Solution

In [ ]:
def categorical_crossentropy(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
  log_probs = torch.log_softmax(logits, dim=-1)
  # For each example i, the log-probability of its right class y[i]
  right_class_log_probs = log_probs[torch.arange(len(y), device=y.device), y]
  return -right_class_log_probs.mean()


# About log(10) = 2.3: the model answers almost at random
logits = SimpleNN()(X_train)
print(categorical_crossentropy(logits, y_train))
# The same value: cross_entropy applies log_softmax itself
print(nn.functional.cross_entropy(logits, y_train))

### Metrics

To know whether a model learns, we measure its performance. We'll use the simplest metric, but also one of the most informative: accuracy, the number of right predictions over the total number of predictions.

*Complete the `accuracy` function. The predicted class is the one with the largest logit (`torch.argmax`), since the softmax doesn't change the scores' order: compare it with the labels, then compute the proportion of right answers.*

**Useful documentation**

- [`torch.argmax`](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  pass  # Your code here

#### Additional hints

- `torch.argmax(y_pred, dim=-1)`: the predicted class for each example
- `predictions == y`: a tensor of booleans
- `.float().mean()`: the proportion of `True`
- `.item()`: a Python number, easier to display and store

Expected: about 0.1 for a model that hasn't learned, one chance in ten.

```python
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(..., dim=...)
  return (... == ...).float().mean().item()


print(accuracy(y_val, SimpleNN()(X_val)))
```

#### Solution

In [ ]:
def accuracy(y: torch.Tensor, y_pred: torch.Tensor) -> float:
  predictions = torch.argmax(y_pred, dim=-1)
  return (predictions == y).float().mean().item()


print(accuracy(y_val, SimpleNN()(X_val)))

## Creating a PyTorch `DataLoader`

The model will learn by mini-batches: at each step, a small group of randomly drawn examples. A `DataLoader` prepares them (slides "Loading Data: `DataLoader`" and "Complete Training Loop").

*Complete the `create_dataset` function, which wraps the tensors `X` and `y` in a `TensorDataset`, then in a `DataLoader` returning batches `(X_batch, y_batch)` of `batch_size` examples, shuffled at each epoch.*

In [ ]:
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  pass  # Your code here

### Additional hints

- `TensorDataset(X, y)`: a dataset whose example `i` is the pair `(X[i], y[i])`
- `DataLoader(dataset, batch_size=batch_size, shuffle=True)`: batches, in an order drawn again at each pass

Expected: `create_dataset(X_train, y_train)` has 391 batches (`len`), and the first one, `next(iter(...))`, has images of shape (128, 784).

```python
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  return DataLoader(TensorDataset(..., ...), batch_size=..., shuffle=...)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)
```

### Solution

In [ ]:
def create_dataset(
  X: torch.Tensor, y: torch.Tensor, batch_size: int = 128
) -> DataLoader:
  # The examples are shuffled at each epoch
  return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)


dataset = create_dataset(X_train, y_train)
X_batch, y_batch = next(iter(dataset))
print(len(dataset), X_batch.shape, y_batch.shape)

## Training

Here is an almost complete training loop: it goes through the batches at each epoch, then measures the loss and the accuracy on the training and validation sets, and plots their curves at the end.

*Implement the missing part: updating the model's parameters for one batch, by gradient descent. Compute the batch's loss, adding `regularization(model)` to it when a regularization is given (it will be used later), reset the gradients to zero, compute them with `backward`, then subtract `learning_rate` times its gradient from each parameter (slides "Automatic Differentiation" and "Disabling Gradient Tracking").*

In [ ]:
LossFunction = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
RegularizationFunction = Callable[[nn.Module], torch.Tensor]


def train(
  model: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train_pred, y_train))]
    val_losses = [float(loss(y_val_pred, y_val))]

  print(
    f"Initial metrics: acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Your code here

    # Compute and display the metrics every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train_pred, y_train)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val_pred, y_val)))
      print(
        f"Epoch {e + 1}: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Additional hints

- `loss(model(X_batch), y_batch)`: the batch's loss; add `regularization(model)` to it if `regularization is not None`
- `model.zero_grad()`: clears the gradients (sets them to `None`), which otherwise accumulate from one `backward` to the next
- `loss_value.backward()`: stores the loss's gradient in the `.grad` attribute of each parameter
- `with torch.no_grad():`: the update must not enter the computation graph
- `parameter -= learning_rate * parameter.grad`: the update, in place, for each parameter of `model.parameters()`

The completed loop over the batches, to copy in place of its own (with its indentation):

```python
for X_batch, y_batch in dataset_train:
  loss_value = ...
  if regularization is not None:
    loss_value = loss_value + ...
  model.zero_grad()
  loss_value.backward()
  with torch.no_grad():
    for parameter in model.parameters():
      parameter -= ...
```

### Solution

In [ ]:
LossFunction = Callable[[torch.Tensor, torch.Tensor], torch.Tensor]
RegularizationFunction = Callable[[nn.Module], torch.Tensor]


def train(
  model: nn.Module,
  X_train: torch.Tensor = X_train,
  y_train: torch.Tensor = y_train,
  X_val: torch.Tensor = X_val,
  y_val: torch.Tensor = y_val,
  epochs: int = 10,
  batch_size: int = 128,
  learning_rate: float = 0.1,
  evaluate_every: int = 1,
  loss: LossFunction = categorical_crossentropy,
  regularization: RegularizationFunction | None = None,
) -> tuple[list[float], ...]:
  with torch.no_grad():
    y_train_pred = model(X_train)
    y_val_pred = model(X_val)
    accuracies = [accuracy(y_train, y_train_pred)]
    val_accuracies = [accuracy(y_val, y_val_pred)]
    losses = [float(loss(y_train_pred, y_train))]
    val_losses = [float(loss(y_val_pred, y_val))]

  print(
    f"Initial metrics: acc {accuracies[-1]:.4f}, "
    f"val_acc {val_accuracies[-1]:.4f}, "
    f"loss {losses[-1]:.4f}, "
    f"val_loss {val_losses[-1]:.4f}"
  )

  dataset_train = create_dataset(X_train, y_train, batch_size)

  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      # Compute the loss, with the regularization penalty if there is one
      loss_value = loss(model(X_batch), y_batch)
      if regularization is not None:
        loss_value = loss_value + regularization(model)

      # Automatic computation of the loss's gradient with respect to each
      # parameter: it is stored in the parameter's .grad attribute
      model.zero_grad()
      loss_value.backward()

      # Update each parameter following the rule:
      # new_value = old_value - learning_rate * gradient
      with torch.no_grad():
        for parameter in model.parameters():
          parameter -= learning_rate * parameter.grad

    # Compute and display the metrics every `evaluate_every` epochs
    if (e + 1) % evaluate_every == 0:
      with torch.no_grad():
        y_train_pred = model(X_train)
        y_val_pred = model(X_val)
        accuracies.append(accuracy(y_train, y_train_pred))
        losses.append(float(loss(y_train_pred, y_train)))
        val_accuracies.append(accuracy(y_val, y_val_pred))
        val_losses.append(float(loss(y_val_pred, y_val)))
      print(
        f"Epoch {e + 1}: acc {accuracies[-1]:.4f}, "
        f"val_acc {val_accuracies[-1]:.4f}, "
        f"loss {losses[-1]:.4f}, "
        f"val_loss {val_losses[-1]:.4f}"
      )

  x_ticks = np.arange(0, epochs + 1, evaluate_every)

  plt.plot(x_ticks, losses, label="Training")
  plt.plot(x_ticks, val_losses, label="Validation")
  plt.title("Loss function during training")
  plt.xlabel("Epochs")
  plt.legend(loc="upper right")
  plt.show()
  plt.plot(x_ticks, accuracies, label="Training")
  plt.plot(x_ticks, val_accuracies, label="Validation")
  plt.title("Accuracy during training")
  plt.xlabel("Epochs")
  plt.legend(loc="lower right")
  plt.show()
  return accuracies, val_accuracies, losses, val_losses

### Training the Network Without a Hidden Layer

10 epochs, with a learning rate of 0.5:

In [ ]:
simple_nn = SimpleNN()
_ = train(simple_nn, epochs=10, learning_rate=0.5)

*What validation accuracy does this model reach? Does it still improve after a few epochs?*

Your answer here

#### Solution

About 92.5% validation accuracy. Most of it is learned in the first epoch (about 91.5%); after that, the model gains only one more point in 9 epochs, oscillating from one epoch to the next. The training and validation accuracies stay equal: the model doesn't overfit, it is too simple. A logistic regression separates the classes with linear boundaries, which plateaus around 92% on MNIST.

## With a PyTorch Optimizer

The update written by hand above (`parameter -= learning_rate * parameter.grad`) is exactly what the `torch.optim.SGD` optimizer does. In practice, we therefore write the standard training loop (slide "Complete Training Loop"), where one step of the optimizer replaces the loop over the parameters.

*Complete the loop of `train_with_optimizer`, which takes one step of `optimizer` per batch: resetting the gradients, the batch's loss, computing the gradients, updating the parameters. The next exercise uses it to compare three optimizers.*

In [ ]:
def train_with_optimizer(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      pass  # Your code here

    with torch.no_grad():
      val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1}: val_acc {val_accuracy:.4f}")

### Additional hints

- The loss is computed as in `train`, with `categorical_crossentropy`, without regularization
- `optimizer.zero_grad()`, `loss_value.backward()`, then `optimizer.step()`: one step of the optimizer on all the parameters it was given when created

The completed loop over the batches, to copy in place of its own (with its indentation):

```python
for X_batch, y_batch in dataset_train:
  optimizer.zero_grad()
  loss_value = categorical_crossentropy(..., ...)
  ...  # compute the gradients
  ...  # update the parameters
```

### Solution

In [ ]:
def train_with_optimizer(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  epochs: int = 5,
  batch_size: int = 128,
) -> None:
  dataset_train = create_dataset(X_train, y_train, batch_size)
  for e in range(epochs):
    for X_batch, y_batch in dataset_train:
      optimizer.zero_grad()
      loss_value = categorical_crossentropy(model(X_batch), y_batch)
      loss_value.backward()
      optimizer.step()

    with torch.no_grad():
      val_accuracy = accuracy(y_val, model(X_val))
    print(f"Epoch {e + 1}: val_acc {val_accuracy:.4f}")

### Comparing Optimizers

*Train three new `SimpleNN` with `train_with_optimizer`, 5 epochs each: the first one with `torch.optim.SGD` and the learning rate of the loop written by hand, 0.5; the second one with SGD, a momentum of 0.9 and a learning rate ten times smaller, 0.05; the third one with `torch.optim.Adam` and its usual learning rate, 10⁻³. Do you get the performance of the loop written by hand with SGD? What do momentum and Adam change?*

**Useful documentation**

- [`torch.optim.SGD`](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

In [ ]:
# Your code here

#### Additional hints

- `torch.optim.SGD(model.parameters(), lr=0.5)`: an optimizer is tied to a model's parameters; a new model needs a new optimizer
- `momentum=0.9`: an argument of `torch.optim.SGD`
- `torch.optim.Adam(model.parameters(), lr=1e-3)`: Adam, with its usual learning rate

A skeleton to copy into the exercise cell, replacing each `...`:

```python
print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=...))

print("SGD with momentum")
simple_nn = SimpleNN()
train_with_optimizer(
  simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=..., momentum=...)
)

print("Adam")
...
```

#### Solution

In [ ]:
print("SGD")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=0.5))

print("SGD with momentum")
simple_nn = SimpleNN()
train_with_optimizer(
  simple_nn, torch.optim.SGD(simple_nn.parameters(), lr=0.05, momentum=0.9)
)

print("Adam")
simple_nn = SimpleNN()
train_with_optimizer(simple_nn, torch.optim.Adam(simple_nn.parameters(), lr=1e-3))

With SGD and the same learning rate, we get the performance of the loop written by hand back (about 92% in 5 epochs): `optimizer.step()` applies the same update rule, and `optimizer.zero_grad()` replaces `model.zero_grad()`. Changing the optimization algorithm now only takes changing the optimizer.

With momentum, SGD follows a moving average of the gradients rather than the batch's gradient alone (`v = 0.9 * v + gradient`, then `parameter -= lr * v`): when the gradients keep the same direction, the step is about $1 / (1 - 0.9) = 10$ times the learning rate. With a learning rate ten times smaller, it therefore reaches the same result; with 0.5, its steps would be too large and the accuracy would oscillate from one epoch to the next. On this simple model, it doesn't speed up learning; it's mostly useful for deep networks, where it damps the gradient's oscillations from one batch to the next.

Adam adapts each parameter's step to the history of its gradients: with a learning rate 500 times smaller, it reaches the same result too. On this simple model, it brings nothing more; its point is that the value 10⁻³ suits most networks, whereas SGD's has to be searched for each one.

## Compiling the Model With `torch.compile`

So far, PyTorch runs each operation as soon as Python calls it (*eager* mode). `torch.compile` returns a compiled version of the model: at its first call, PyTorch captures its computations in a graph, which TorchInductor compiles into code optimized for the CPU or the GPU, fusing operations. The following calls run this compiled code, as long as their inputs look like those of the compilation (same shape, same `dtype`): an input of another shape triggers a recompilation (slide "Compiling the Model: `torch.compile`").

The next cell creates `model`, a new `SimpleNN`, its SGD optimizer, and `time_epoch(network, name)`, which runs one training epoch of `network` and displays the duration of its training steps: their total, that of the first batch and that of the last one. It times one epoch in eager mode.

*Compile `model` with `torch.compile`, keeping the name `model`, then time two epochs with it. Why is the first batch of the first epoch so slow? And its last batch? Once compiled, does the model train faster than in eager mode?*

**Useful documentation**

- [`torch.compile`](https://docs.pytorch.org/docs/stable/generated/torch.compile.html)

In [ ]:
model = SimpleNN()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
dataset_train = create_dataset(X_train, y_train)


def time_epoch(network: nn.Module, name: str) -> None:
  durations = []
  for X_batch, y_batch in dataset_train:
    start = time.perf_counter()
    optimizer.zero_grad()
    loss_value = categorical_crossentropy(network(X_batch), y_batch)
    loss_value.backward()
    optimizer.step()
    durations.append(time.perf_counter() - start)
  print(
    f"{name}: {sum(durations):.2f} s, first batch {durations[0]:.3f} s, "
    f"last batch {durations[-1]:.3f} s"
  )


time_epoch(model, "Eager")

In [ ]:
# Your code here

### Additional hints

- `model = torch.compile(model)`: the compiled model, to call like the original one; it shares its parameters, and the optimizer created before compiling remains valid
- `time_epoch(model, "Compiled, epoch 1")`: the duration of an epoch with the compiled model

A skeleton to copy into the exercise cell, replacing each `...`:

```python
model = torch.compile(...)
time_epoch(..., "Compiled, epoch 1")
time_epoch(..., "Compiled, epoch 2")
```

### Solution

In [ ]:
# The compiled model shares the original's parameters: the optimizer remains
# valid
model = torch.compile(model)
# The first batch compiles the model, the last one, smaller, recompiles it
time_epoch(model, "Compiled, epoch 1")
time_epoch(model, "Compiled, epoch 2")

The first batch takes several seconds, against a few milliseconds in eager mode: that's the compilation. PyTorch captures the model's computations and those of their gradients, then TorchInductor generates C++ code for the CPU and compiles it. The last batch, of 80 examples (50,000 = 390 $\times$ 128 + 80), has another shape than the others: it triggers a recompilation, of a second or two. PyTorch then compiles a version where the batch size is free: a third size wouldn't recompile anymore. At the second epoch, nothing is compiled anymore. `drop_last=True` in the `DataLoader` would avoid the recompilation, at the cost of the last 80 examples of each epoch.

Once compiled, the model doesn't train faster on the CPU: an epoch takes about as long as in eager mode, sometimes a bit longer. `SimpleNN` only does a matrix product and an addition, there is little to fuse, and each call of the compiled model pays a fixed overhead, such as checking that its inputs match the compiled version. The seconds of compilation are never recovered. The gain comes on GPU, with larger models: the fused operations avoid round trips to memory there. On CPU, measure before keeping the compilation.

The compiled model wraps the original one: the names in its `state_dict` start with `_orig_mod.`. To save the weights, take the `state_dict` of the original, `model._orig_mod.state_dict()`.

## Moving On to Deep Networks

To move on to a multilayer perceptron, we create a weight matrix and a bias vector per layer, with `create_simple_nn_weights`. The following function stores them in an `nn.ModuleList`, one `nn.ParameterDict` `{"W": ..., "b": ...}` per layer: these two containers let `nn.Module` find all the parameters (`model.parameters()`).

In [ ]:
def create_mlp_weights(
  n_inputs: int, n_outputs: int, hidden_layer_sizes: tuple[int, ...] = (512,)
) -> nn.ModuleList:
  layers = nn.ModuleList()
  # A layer's input size is the previous layer's output size
  last_size = n_inputs
  for size in [*hidden_layer_sizes, n_outputs]:
    W, b = create_simple_nn_weights(last_size, size)
    layers.append(nn.ParameterDict({"W": W, "b": b}))
    last_size = size
  return layers

The model's output is now computed layer by layer: each layer takes the previous one's result as input.

*Complete the `forward` method of `MLP`. Each hidden layer computes $\tanh(XW + b)$ with `torch.tanh`; the last one returns the logits $XW + b$, without an activation function, like `SimpleNN`. The parameters of layer `i` are `self.layers[i]["W"]` and `self.layers[i]["b"]`.*

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    pass  # Your code here

### Additional hints

- `self.layers[:-1]`: the hidden layers; `self.layers[-1]`: the output layer
- `torch.tanh(X @ layer["W"] + layer["b"])`: a hidden layer

Expected: `MLP()(X_train[:5])` has shape (5, 10), and `MLP((256, 128))` has 6 parameters (`len(list(....parameters()))`).

```python
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    for layer in self.layers[:-1]:
      X = torch.tanh(...)
    output_layer = self.layers[-1]
    return ... @ output_layer["W"] + ...
```

### Solution

In [ ]:
class MLP(nn.Module):
  def __init__(
    self,
    hidden_layer_sizes: tuple[int, ...] = (512,),
    n_inputs: int = 28**2,
    n_outputs: int = 10,
  ) -> None:
    super().__init__()
    self.layers = create_mlp_weights(n_inputs, n_outputs, hidden_layer_sizes)

  def forward(self, X: torch.Tensor) -> torch.Tensor:
    # Hidden layers: affine transformation, then tanh
    for layer in self.layers[:-1]:
      X = torch.tanh(X @ layer["W"] + layer["b"])
    # Output layer: the logits, without activation
    output_layer = self.layers[-1]
    return X @ output_layer["W"] + output_layer["b"]


print(MLP()(X_train[:5]).shape, len(list(MLP((256, 128)).parameters())))

## Training the Deep Network

One hidden layer of 512 neurons, 10 epochs, a learning rate of 0.5:

In [ ]:
mlp = MLP()
_ = train(mlp, epochs=10, learning_rate=0.5)

*What do you notice when comparing this network with the network without a hidden layer?*

Your answer here

### Solution

The network with one hidden layer does much better than the network without a hidden layer: about 97.5% validation accuracy, versus 92.5%, an error divided by 3. The hidden layer and its tanh make the model nonlinear. The training accuracy gets close to 99% and moves away from the validation one: the network starts to overfit, but its validation loss still goes down. With a learning rate of 0.5, the curves aren't monotonic: one epoch can lose a point or two, which the next ones win back.

This is partly thanks to the weight initialization, with a variance of $1 / n_\text{inputs}$. To convince yourself, remove the division by `n_inputs**0.5` in `create_simple_nn_weights` and run that cell and the training again: with weights of variance 1, three quarters of the tanh activations are saturated (above 0.99 in absolute value) from the start, the initial logits are huge (a loss of about 31 instead of 2.3), and the network reaches 99.8% on training but only 93% on validation. This large gap looks like overfitting, but it comes from a bad initialization (slide "Weight Initialization").

## Regularization

The previous perceptron overfits little: 50,000 examples are enough for its 400,000 weights. Overfitting appears when the network is large compared with the data: the cell after the solution trains the same perceptron on the first 1,000 examples only, 100 epochs, without then with regularization.

L2 regularization adds a penalty to the loss, $\lambda \sum w^2$: $\lambda$ times the sum of the squared weights, which pushes the weights towards 0 (slide "Via Weight Penalization"). Only the weights `W` are penalized, not the biases: they don't multiply the input, and constraining them doesn't help generalization.

*Complete the inner function of `l2`, which returns the penalty of an `MLP`: the sum of the squares of the `W` matrices of all its layers (`model.layers`), multiplied by `lambda_coefficient`.*

**Useful documentation**

- [`torch.sum`](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    pass  # Your code here

  return penalty

### Additional hints

- `torch.sum(layer["W"] ** 2)`: the sum of a matrix's squares
- Python's `sum(...)`, over all the layers of `model.layers`, to add them up

Expected: for a new `MLP()`, `l2(1.0)(MLP())` is about 522 (512 for the first layer, 10 for the second one: each contributes roughly its number of columns, since each weight has a variance of $1 / n_\text{inputs}$).

```python
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    weights_norm = sum(torch.sum(...) for layer in model.layers)
    return ... * weights_norm

  return penalty
```

### Solution

In [ ]:
def l2(lambda_coefficient: float) -> RegularizationFunction:
  def penalty(model: MLP) -> torch.Tensor:
    # The W weights only, not the biases
    weights_norm = sum(torch.sum(layer["W"] ** 2) for layer in model.layers)
    return lambda_coefficient * weights_norm

  return penalty


# .item(): the value, as a Python number, outside the autograd graph
print(l2(1.0)(MLP()).item())

### Overfitting on 1,000 Examples

The same perceptron, with the same learning rate of 0.5, on the first 1,000 training examples, without regularization then with an L2 penalty of $\lambda$ = 10⁻³; validation is still done on the 10,000 examples set aside.

In [ ]:
X_small, y_small = X_train[:1000], y_train[:1000]

print("Without regularization")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
)

print("With an L2 penalty")
_ = train(
  MLP(),
  X_train=X_small,
  y_train=y_small,
  epochs=100,
  learning_rate=0.5,
  evaluate_every=10,
  regularization=l2(1e-3),
)

*Compare the two training runs: the gap between training and validation, the validation loss over the epochs, the final validation accuracy.*

Your answer here

#### Solution

Without regularization, the network learns the 1,000 examples by heart: 100% training accuracy from the thirtieth epoch on, a training loss that tends to 0, and a validation accuracy stuck around 87%. The validation loss is at its lowest within the first 10 epochs, then keeps going up, from 0.44 to 0.60: the network becomes more and more confident in its answers, wrong ones included. This is overfitting.

With the L2 penalty, the training loss stays above 0.027 (the weights can no longer grow freely), and the validation loss goes up much less: it levels off around 0.47. The validation accuracy, for its part, doesn't change: on so little data, regularization mostly limits the model's overconfidence. Nothing replaces more data: the same network reaches 97.5% with the 50,000 examples.

## Following Training With TensorBoard

The curves `train` plots disappear with the notebook, and are hard to compare from one training run to another. TensorBoard displays logs, written during training:

- `SummaryWriter(log_dir)`, from `torch.utils.tensorboard`, creates a log in `log_dir`;
- `writer.add_scalar(name, value, epoch)` adds a point to the `name` curve;
- `writer.close()` writes to disk what is still in memory.

One directory per run: TensorBoard overlays their curves, which lets us compare hyperparameters on the validation set, never on the test set. Comparing a few values of one hyperparameter is a grid search (slide "Hyperparameter Search: Grid or Random?").

*Complete `train_with_log`: at each epoch, take one step of `optimizer` per batch, as in `train_with_optimizer`, then write to the log the mean loss of the epoch (`loss`) and the validation accuracy (`val_accuracy`). The function returns the last validation accuracy. The cell after the solution trains an MLP by SGD with three learning rates, each in its own directory of `logs/`, then opens TensorBoard.*

**Useful documentation**

- [`SummaryWriter`](https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter)
- [`add_scalar`](https://docs.pytorch.org/docs/stable/tensorboard.html#torch.utils.tensorboard.writer.SummaryWriter.add_scalar)

In [ ]:
def train_with_log(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  pass  # Your code here

### Additional hints

- `writer = SummaryWriter(log_dir)` at the start of the function, `writer.close()` at the end
- The loop over the batches of `train_with_optimizer`; `loss_value.item()`, appended to a list `losses`, and `np.mean(losses)`: the mean loss of the epoch
- `accuracy(y_val, model(X_val))`, under `torch.no_grad()`: the validation accuracy
- `writer.add_scalar("loss", ..., e + 1)`: the point's x-coordinate, in epochs

```python
def train_with_log(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  writer = SummaryWriter(...)
  dataset_train = create_dataset(X_train, y_train)
  for e in range(epochs):
    losses = []
    for X_batch, y_batch in dataset_train:
      ...  # one step of the optimizer, as in train_with_optimizer
      losses.append(loss_value.item())
    with torch.no_grad():
      val_accuracy = accuracy(..., ...)
    writer.add_scalar("loss", ..., e + 1)
    writer.add_scalar(..., ..., ...)
  writer.close()
  return val_accuracy
```

### Solution

In [ ]:
def train_with_log(
  model: nn.Module,
  optimizer: torch.optim.Optimizer,
  log_dir: str,
  epochs: int = 5,
) -> float:
  writer = SummaryWriter(log_dir)
  dataset_train = create_dataset(X_train, y_train)
  for e in range(epochs):
    losses = []
    for X_batch, y_batch in dataset_train:
      optimizer.zero_grad()
      loss_value = categorical_crossentropy(model(X_batch), y_batch)
      loss_value.backward()
      optimizer.step()
      losses.append(loss_value.item())
    with torch.no_grad():
      val_accuracy = accuracy(y_val, model(X_val))
    # One point per curve and per epoch
    writer.add_scalar("loss", np.mean(losses), e + 1)
    writer.add_scalar("val_accuracy", val_accuracy, e + 1)
    print(f"Epoch {e + 1}: val_acc {val_accuracy:.4f}")
  writer.close()
  return val_accuracy

### Comparing Three Learning Rates

One MLP per learning rate, 5 epochs each, logged to `logs/sgd_<learning rate>`:

In [ ]:
mlps, val_accuracies = {}, {}
for rate in (0.01, 0.1, 0.5):
  print(f"Learning rate {rate}")
  mlps[rate] = MLP()
  optimizer = torch.optim.SGD(mlps[rate].parameters(), lr=rate)
  val_accuracies[rate] = train_with_log(mlps[rate], optimizer, f"logs/sgd_{rate}")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

*In TensorBoard, which learning rate gives the best validation accuracy? What do the loss curves show?*

Your answer here

#### Solution

0.5: about 96.5 to 97% validation accuracy in 5 epochs, against 93.5% for 0.1 and 90% for 0.01 (`val_accuracy` curve). The `loss` curves show why: the smaller the learning rate, the more slowly the training loss goes down. At 0.01, the network is far from done learning in 5 epochs: comparing such short runs favors what learns fast, not necessarily what ends best.

TensorBoard reads the logs again as they are written: you can follow a training run in progress, with some delay, as `SummaryWriter` only writes to disk every two minutes or every ten points (`flush_secs` and `max_queue`). Running the cell again adds new logs to the old ones, in the same directories: delete `logs/` to start from scratch.

## Final Evaluation on the Test Set

The test set has been used for nothing so far: neither for training, nor for choosing the learning rate. That's what makes its evaluation reliable.

*Take `mlp`, the MLP trained with the learning rate of the best validation accuracy, and compute its accuracy on `X_test`. Is it close to its validation accuracy?*

In [ ]:
# Your code here

### Additional hints

- `max(val_accuracies, key=val_accuracies.get)`: the key of the dictionary's largest value, here the best learning rate
- `mlps[best_rate]`: the MLP trained with it
- `accuracy(y_test, mlp(X_test))`, under `torch.no_grad()`: the test accuracy

```python
best_rate = max(..., key=...)
mlp = mlps[...]
with torch.no_grad():
  print(f"Accuracy on the test set: {accuracy(..., ...):.4f}")
```

### Solution

In [ ]:
# The learning rate of the best validation accuracy, and its MLP
best_rate = max(val_accuracies, key=val_accuracies.get)
mlp = mlps[best_rate]
print(f"Chosen learning rate: {best_rate}")
print(f"Accuracy on the validation set: {val_accuracies[best_rate]:.4f}")
with torch.no_grad():
  print(f"Accuracy on the test set: {accuracy(y_test, mlp(X_test)):.4f}")

The test accuracy is close to the validation accuracy, within half a point (for example 96.6% versus 97.1%). The validation set was used to choose the learning rate: its accuracy is in principle slightly optimistic, all the more so as more runs were compared. The test set, which wasn't used for any choice, gives a reliable estimate of the model's performance on new data.

With a learning rate of 0.5, the last epoch isn't always the best one: the validation accuracy can go down from one epoch to the next, by a few tenths of a point over these 5 epochs, by one or two points over the 10 of "Training the Deep Network". Keeping the weights of the best validation epoch is early stopping (slide "Via Early Stopping").

## Saving and Loading the Model

Training is costly: we save the trained model to reuse it without training it again. We save its `state_dict`, the dictionary of its parameters stored by name, rather than the Python object itself (slide "Saving and Loading a Model").

*Save the `state_dict` of `mlp` to the file `mlp.pt`. Create `restored`, a new `MLP` with the same architecture, and display its test accuracy before and after loading these weights into it.*

In [ ]:
# Your code here

### Additional hints

- `torch.save(mlp.state_dict(), "mlp.pt")`: the save; the keys of the `state_dict` are the parameters' attribute paths (`layers.0.W`, `layers.0.b`…)
- `restored` must have the same architecture as `mlp`, an `MLP()` with its default sizes: same parameter names, same shapes
- `torch.load("mlp.pt", map_location=device)`: reads the dictionary back, its tensors placed on `device`; `restored.load_state_dict(...)` copies its values into the parameters of `restored`
- `accuracy(y_test, restored(X_test))`, under `torch.no_grad()`: the test accuracy

Expected: about 0.1 before loading, exactly the accuracy of `mlp` after.

```python
torch.save(..., "mlp.pt")
restored = ...
with torch.no_grad():
  print(accuracy(y_test, restored(X_test)))
restored.load_state_dict(torch.load(..., map_location=...))
with torch.no_grad():
  print(accuracy(..., ...))
```

### Solution

In [ ]:
torch.save(mlp.state_dict(), "mlp.pt")
# The keys: each parameter's attribute path
print(list(mlp.state_dict().keys()))

# New random weights, same architecture as mlp
restored = MLP()
with torch.no_grad():
  print(f"Before loading: {accuracy(y_test, restored(X_test)):.4f}")
restored.load_state_dict(torch.load("mlp.pt", map_location=device))
with torch.no_grad():
  print(f"After loading: {accuracy(y_test, restored(X_test)):.4f}")
  print(f"Saved model: {accuracy(y_test, mlp(X_test)):.4f}")

Before loading, `restored` answers almost at random (around 10%, its weights are random). After, it has exactly the accuracy of `mlp`: `load_state_dict` copied each tensor into the parameter with the same name (`layers.0.W`, `layers.0.b`, `layers.1.W`, `layers.1.b`). Hence the importance of an identical architecture: an `MLP` of another size would have parameters of different shapes, and the loading would fail. `map_location` places the tensors read on `device`: without it, weights saved on a GPU don't load on a machine that has none.

The file only holds values, not the model's code: to load it back, you need the `MLP` class. Our `MLP` has neither dropout nor batch normalization; for a model that has them, call `restored.eval()` before predicting. And to resume training, also save `optimizer.state_dict()` and the epoch number.